# Silver Processing — Products

Cleans `ecommerce.bronze.bronze_products` (50,000 rows) into `ecommerce.silver.silver_products`:

* **All business columns**: trim spaces and strip trailing special characters
* **`weight_grams`**: strip the `g` suffix, cast to `int` (all values are whole numbers)
* **`length_cm`**: replace comma decimals (`22,2`) with dot decimals, cast to `double`
* **`brand_code` / `category_code`**: uppercase to join with `silver_brands` / `silver_category`
* **`material`**: fix misspellings — `Coton`→`Cotton`, `Ruber`→`Rubber`, `Alumium`→`Aluminum`
* **`rating_count`**: 659 negative values (down to -203) clamped to `0` — negatives treated as invalid, magnitude discarded
* **Metadata columns** (`source_file`, `ingested_at`) pass through untouched

In [0]:
from pyspark.sql.functions import col, trim, upper, when, greatest, lit, regexp_replace

bronze_table = "ecommerce.bronze.bronze_products"
target_table = "ecommerce.silver.silver_products"

df_bronze_products = spark.table(bronze_table)

# Clean a business column: trim spaces and strip any trailing special characters
def clean(column_name):
    return regexp_replace(trim(col(column_name)), "[^A-Za-z0-9]+$", "")

df_silver_products = (
    df_bronze_products
    # String columns: trim + strip trailing special characters
    .withColumn("sku", clean("sku"))
    .withColumn("color", clean("color"))
    .withColumn("size", clean("size"))
    # Codes to CAPS so they join with silver_brands / silver_category
    .withColumn("brand_code", upper(clean("brand_code")))
    .withColumn("category_code", upper(clean("category_code")))
    # Fix material misspellings
    .withColumn(
        "material",
        when(clean("material") == "Coton", "Cotton")
        .when(clean("material") == "Ruber", "Rubber")
        .when(clean("material") == "Alumium", "Aluminum")
        .otherwise(clean("material")),
    )
    # weight_grams: strip 'g' suffix -> int (all values are whole numbers)
    .withColumn("weight_grams", regexp_replace(trim(col("weight_grams")), "g$", "").cast("int"))
    # length_cm: comma decimal -> dot decimal -> double
    .withColumn("length_cm", regexp_replace(trim(col("length_cm")), ",", ".").cast("double"))
    # rating_count: negative counts are invalid -> clamp to 0
    .withColumn("rating_count", greatest(col("rating_count"), lit(0)).cast("int"))
)

# Idempotent full load into the silver table
(
    df_silver_products.write
    .mode("overwrite")
    .saveAsTable(target_table)
)

print(f"Wrote {spark.table(target_table).count()} rows to {target_table}")

In [0]:
display(spark.table("ecommerce.silver.silver_products").limit(10))

In [0]:
# Check 1: cleaned types and ranges — no negatives, weight int, length double
spark.sql("""
    SELECT
      MIN(weight_grams) AS min_w, MAX(weight_grams) AS max_w,
      MIN(length_cm) AS min_l, MAX(length_cm) AS max_l,
      MIN(rating_count) AS min_rating, MAX(rating_count) AS max_rating,
      COUNT_IF(rating_count < 0) AS neg_ratings
    FROM ecommerce.silver.silver_products
""").show()

# Check 2: corrected materials (no misspellings left)
spark.sql("""
    SELECT material, COUNT(*) AS cnt
    FROM ecommerce.silver.silver_products
    GROUP BY material
    ORDER BY material
""").show(20, truncate=False)

# Check 3: every brand_code and category_code resolves against the silver dimension tables
spark.sql("""
    SELECT
      (SELECT COUNT(*) FROM ecommerce.silver.silver_products p
        LEFT JOIN ecommerce.silver.silver_brands b ON p.brand_code = b.brand_code
        WHERE b.brand_code IS NULL) AS unmatched_brands,
      (SELECT COUNT(*) FROM ecommerce.silver.silver_products p
        LEFT JOIN ecommerce.silver.silver_category c ON p.category_code = c.category_code
        WHERE c.category_code IS NULL) AS unmatched_categories
""").show()